In [1]:
#README


In [2]:
import pandas as pd

#IMPORTS

from functions import *





ModuleNotFoundError: No module named 'yfinance'

In [ ]:
#INPUTS

target_ticker = "ADS.DE"

benchmark = "URTH"
start_date = "2020-12-31"
end_date = "2026-08-01"
interval = "1wk"
return_calc = "linear" #linear / log
beta_adjustment = "blume" #blume / vasicek / none
peer_group_beta_method = "median" #average / median
rf_lookback_months = 1 #from valuation date or latest available data
equity_risk_premium = 0.055
size_premium = 0.015
comp_spec_risk_premium = 0.00
target_longt_sp_rating = "A"
debt_spread_lookback_months = 1 #from valuation date or latest available data

peer_group = ["NKE", "PUM.DE", "ONON", "DECK", "CROX"]


# Ce = rf + beta * mrp + sp +csrp
# Cd = rf + debt spread

In [ ]:
#CLOSE_DATA_COLLECTION

ticker_package = peer_group + [benchmark]
data_package = download_data(tickers= ticker_package, start_date= start_date, end_date= end_date, interval= interval)
save_data(data = data_package, benchmark= benchmark, peer_group= peer_group, start_date= start_date, end_date= end_date)
close_data = extract_col(data = data_package, field= "Close")



In [ ]:
#BASIC_DATA_CLEANING
basic_cleaning(close_data=close_data)

In [ ]:
#LOG_RETURN_CALC
return_data = log_return_calc(return_calc=return_calc, close_data= close_data)



In [ ]:
#BETA_REGRESSION
beta_results = beta_regression(peer_group=peer_group, return_data=return_data, benchmark=benchmark)

In [ ]:
beta_adj = beta_adjustments(beta_results)


In [ ]:
beta_res = append_d_e_ratio(beta_results=beta_results, end_date= end_date)

In [ ]:
#COUNTRY + STATUTORY_TAX_RATE

country_list_2 = []
stat_tax_rates_2 = []

for ticker in beta_results["Ticker"]:
    code = get_country_code_a1(ticker)
    country_list_2.append(code)

beta_results["Country code_2"] = country_list_2

for ticker in beta_results["Ticker"]:
    stat_rate = get_stat_tax_rate(ticker)
    stat_tax_rates_2.append(stat_rate)

beta_results["Statutory tax rate"] = stat_tax_rates_2

beta_results


In [ ]:
def append_tax_rates(beta_results:pd.DataFrame):
    country_list_2 = []
    stat_tax_rates_2 = []

    for ticker in beta_results["Ticker"]:
        code = get_country_code_a1(ticker)
        country_list_2.append(code)

    beta_results["Country code_2"] = country_list_2

    for ticker in beta_results["Ticker"]:
        stat_rate = get_stat_tax_rate(ticker)
        stat_tax_rates_2.append(stat_rate)

    beta_results["Statutory tax rate"] = stat_tax_rates_2

    beta_results

In [ ]:
ad = append_tax_rates(beta_results=beta_res)
ad

In [ ]:
#UNLEVERED BETA (HAMADA)

if beta_adjustment == "none":
    applied_beta = "Raw Betas"
elif beta_adjustment == "blume":
    applied_beta = "Blume adj. beta"
elif beta_adjustment == "vasicek":
    applied_beta = "Vasicek adj. beta"
else:
    raise ValueError("Beta adjustment must be either 'none' / 'blume' / 'vasicek'")

unlevered_beta_list = []

for ticker in beta_results["Ticker"]:
    levered_b = beta_results.loc[beta_results["Ticker"] == ticker, applied_beta].item()
    tax_rate = beta_results.loc[beta_results["Ticker"] == ticker, "Statutory tax rate"].item()
    peer_d_e_ratio = beta_results.loc[beta_results["Ticker"] == ticker, "D/E ratio"].item()
    unlevered_beta = levered_b /(1+(1-tax_rate)*peer_d_e_ratio)
    unlevered_beta_list.append(unlevered_beta)

beta_results["Unlevered beta"] = unlevered_beta_list

if peer_group_beta_method == "average":
    peer_group_beta = beta_results["Unlevered beta"].mean()
elif peer_group_beta_method == "median":
    peer_group_beta = beta_results["Unlevered beta"].median()
else:
    raise ValueError("Peer group beta method must be either 'average' or 'median'")




In [ ]:
target_levered_beta = get_target_levered(target=target_ticker, end_date=end_date, peer_group_beta=peer_group_beta)

In [ ]:
target_rf_rate = target_rf(target=target_ticker, BASE_STR_1=BASE_STRING_1, BASE_STR_2=BASE_STRING_2,start_date=start_date, end_date=end_date, rf_lookback=rf_lookback_months)

In [ ]:
spread_series = get_rating_spread_series(rating_spread_series=RATING_SPREAD_SERIES, start_date=start_date, end_date=end_date)

In [ ]:
interpolated_series = interpolate_spreads(spread_series)

In [ ]:
get_target_spread(interpolated_series, target_longt_sp_rating, lookback=debt_spread_lookback_months)